In [ ]:
import pandas as pd
from IPython.display import display
df = pd.read_csv('On_Time_Marketing_Carrier_On_Time_Performance_(Beginning_January_2018)_2023_1.csv', nrows=500)
total = len(df)
null_counts = df.isnull().sum()
def pct(n, total):
    return round((n / total) * 100, 2) if total else 0.0
issues = []
if 'Operating_Airline ' in df.columns:
    issues.append({
        'Columna': 'Operating_Airline ',
        'Descripcion': 'Nombre de aerolinea operadora con espacio al final',
        'Evidencia': "Columna existe con espacio en el nombre",
        'Razon': 'Nombres inconsistentes rompen joins y renombre...'
    })
if 'Unnamed: 119' in df.columns:
    issues.append({
        'Columna': 'Unnamed: 119',
        'Descripcion': 'Columna sin nombre generada por el CSV',
        'Evidencia': 'Columna extra sin significado',
        'Razon': 'Artefacto de exportacion; no aporta informacion'
    })
div_cols = [c for c in df.columns if c.startswith('Div1') or c.startswith('Div2') or c.startswith('Div3') or c.startswith('Div4') or c.startswith('Div5')]
if div_cols:
    div_null_pct = round(null_counts[div_cols].mean() / total * 100, 2) if total else 0.0
    issues.append({
        'Columna': 'Div1-Div5 (grupo)',
        'Descripcion': 'Campos de desvio (Diverted) por aeropuertos al...',
        'Evidencia': f"{len(div_cols)} columnas con ~{div_null_pct}% nulos en promedio",
        'Razon': 'Solo aplica a vuelos desviados; casi siempre nulos'
    })
if 'Cancelled' in df.columns and 'CancellationCode' in df.columns:
    mask_cancelled_zero = (df['Cancelled'] == 0)
    null_cancel_code = df.loc[mask_cancelled_zero, 'CancellationCode'].isnull().sum()
    issues.append({
        'Columna': 'CancellationCode',
        'Descripcion': 'Codigo de cancelacion',
        'Evidencia': f"Nulos cuando Cancelled=0: {null_cancel_code}/{mask_cancelled_zero.sum()} ({pct(null_cancel_code, mask_cancelled_zero.sum())}%)",
        'Razon': 'Campo solo aplica a vuelos cancelados; nulo cu...'
    })
if 'Cancelled' in df.columns:
    mask_cancelled_one = (df['Cancelled'] == 1)
    for col in ['ArrDelay', 'DepDelay']:
        if col in df.columns:
            nulls = df.loc[mask_cancelled_one, col].isnull().sum()
            issues.append({
                'Columna': col,
                'Descripcion': f"Retraso de {col.replace('Delay', '')} en minutos",
                'Evidencia': f"Nulos cuando Cancelled=1: {nulls}/{mask_cancelled_one.sum()} ({pct(nulls, mask_cancelled_one.sum())}%)",
                'Razon': 'Retrasos no existen si el vuelo se cancela'
            })
if 'FlightDate' in df.columns:
    dtype = df['FlightDate'].dtype
    if str(dtype) == 'object':
        issues.append({
            'Columna': 'FlightDate',
            'Descripcion': 'Fecha del vuelo',
            'Evidencia': f"Tipo actual: {dtype}",
            'Razon': 'Debe ser fecha para ordenar y filtrar correctamente'
        })
issues_df = pd.DataFrame(issues)
display(issues_df)
display(null_counts.sort_values(ascending=False).head(15).rename('Nulos').to_frame())

,Columna,Descripcion,Evidencia,Razon
0,Operating_Airline,Nombre de aerolinea operadora con espacio al f...,Columna existe con espacio en el nombre,Nombres inconsistentes rompen joins y renombre...
1,Unnamed: 119,Columna sin nombre generada por el CSV,Columna extra sin significado,Artefacto de exportacion; no aporta informacion
2,Div1-Div5 (grupo),Campos de desvio (Diverted) por aeropuertos al...,40 columnas con ~99.92% nulos en promedio,Solo aplica a vuelos desviados; casi siempre n...
3,CancellationCode,Codigo de cancelacion,Nulos cuando Cancelled=0: 499/499 (100.0%),Campo solo aplica a vuelos cancelados; nulo cu...
4,ArrDelay,Retraso de Arr en minutos,Nulos cuando Cancelled=1: 1/1 (100.0%),Retrasos no existen si el vuelo se cancela
5,DepDelay,Retraso de Dep en minutos,Nulos cuando Cancelled=1: 1/1 (100.0%),Retrasos no existen si el vuelo se cancela


,Nulos
DOT_ID_Originally_Scheduled_Code_Share_Airline,500
IATA_Code_Originally_Scheduled_Code_Share_Airline,500
Flight_Num_Originally_Scheduled_Code_Share_Airline,500
Originally_Scheduled_Code_Share_Airline,500
Div3WheelsOff,500
Div3TailNum,500
Div4Airport,500
Div4AirportID,500
Div4AirportSeqID,500
Div4WheelsOn,500
